# Topic 4: Data Cleaning and Quality Checks

This notebook prepares the data for M1 exploration. It loads all ten full data files, reports counts before and after each step, checks relationships between files, and records evidence for corrections. It never overwrites the raw files.


In [1]:
from pathlib import Path
from io import BytesIO
import os
import json
import hashlib
import zipfile
import pandas as pd

project_root = Path.cwd().parent if Path.cwd().name in {"notebooks", "scripts"} else Path.cwd()
raw_dir = Path(os.environ.get("TOPIC4_RAW_DIR", str(project_root / "Data" / "data")))
clean_dir = Path(os.environ.get("TOPIC4_CLEAN_DIR", str(project_root / "cleaned_data")))
report_dir = Path(os.environ.get("TOPIC4_REPORT_DIR", str(project_root / "cleaning_reports")))
if raw_dir.resolve() == clean_dir.resolve():
    raise ValueError("Raw and cleaned directories must be different.")
clean_dir.mkdir(parents=True, exist_ok=True)
report_dir.mkdir(parents=True, exist_ok=True)
steps = []
checks = []

def record(file, step, before, after, affected, reason):
    row = {"file": file, "step": step, "rows_before": before, "rows_after": after,
           "affected_rows": int(affected), "reason": reason}
    steps.append(row)
    print(f"{file} | {step}: {before:,} -> {after:,} rows; affected={affected:,}. {reason}")

def check(label, condition):
    checks.append({"check": label, "passed": bool(condition)})
    if not condition:
        raise ValueError(f"Quality check failed: {label}")

print("Raw data:", raw_dir)
print("Cleaned data:", clean_dir)
print("Reports:", report_dir)


Raw data: c:\Users\lsadi\Documents\GitHub\CSC-4913-Project-4\Data\data
Cleaned data: c:\Users\lsadi\Documents\GitHub\CSC-4913-Project-4\cleaned_data
Reports: c:\Users\lsadi\Documents\GitHub\CSC-4913-Project-4\cleaning_reports


### 1. Load the full files and remove exact duplicates

We load the full datasets, excluding the sample files. We remove rows that are identical in every column and report how many were removed from each file. If two rows identify the same game but contain different values, we flag them for investigation instead of choosing one automatically.


In [2]:
steps.clear()
names = ["teams", "conferences", "team_conferences", "seasons", "regular_season_results",
         "box_scores", "conference_tourney_games", "tournament_seeds", "tournament_results",
         "team_strength_preseason"]
raw = {}
clean = {}
inventory = []
for name in names:
    paths = [raw_dir / (name + suffix) for suffix in [".csv", ".csv.gz", ".csv.zip"]]
    paths = [path for path in paths if path.exists()]
    if len(paths) != 1:
        raise ValueError(f"Expected one full input for {name}; found {paths}")
    path = paths[0]
    if path.suffix == ".zip":
        with zipfile.ZipFile(path) as archive:
            members = [m for m in archive.namelist() if m.endswith(".csv") and not m.startswith("__MACOSX/")]
            check(f"One CSV inside {path.name}", len(members) == 1)
            frame = pd.read_csv(archive.open(members[0]))
    else:
        frame = pd.read_csv(path)
    raw[name] = frame
    inventory.append({"file": name, "source_filename": path.name, "raw_rows": len(frame),
                      "sha256": hashlib.sha256(path.read_bytes()).hexdigest()})
    record(name, "load", len(frame), len(frame), 0, "Read the full supplied file; sample files are not used.")
    clean[name] = frame.drop_duplicates(ignore_index=True).copy()
    record(name, "remove exact duplicates", len(frame), len(clean[name]), len(frame)-len(clean[name]),
           "Remove only fully identical rows.")

duplicate_summary = pd.DataFrame(steps)
duplicate_summary = duplicate_summary.loc[
    duplicate_summary["step"] == "remove exact duplicates",
    ["file", "rows_before", "affected_rows", "rows_after"]
].rename(columns={
    "file": "File",
    "rows_before": "Rows loaded",
    "affected_rows": "Duplicates removed",
    "rows_after": "Rows remaining"
})

display(duplicate_summary.reset_index(drop=True))


teams | load: 360 -> 360 rows; affected=0. Read the full supplied file; sample files are not used.
teams | remove exact duplicates: 360 -> 360 rows; affected=0. Remove only fully identical rows.
conferences | load: 32 -> 32 rows; affected=0. Read the full supplied file; sample files are not used.
conferences | remove exact duplicates: 32 -> 32 rows; affected=0. Remove only fully identical rows.
team_conferences | load: 4,298 -> 4,298 rows; affected=0. Read the full supplied file; sample files are not used.
team_conferences | remove exact duplicates: 4,298 -> 4,298 rows; affected=0. Remove only fully identical rows.
seasons | load: 12 -> 12 rows; affected=0. Read the full supplied file; sample files are not used.
seasons | remove exact duplicates: 12 -> 12 rows; affected=0. Remove only fully identical rows.
regular_season_results | load: 65,670 -> 65,670 rows; affected=0. Read the full supplied file; sample files are not used.
regular_season_results | remove exact duplicates: 65,670 -> 

,File,Rows loaded,Duplicates removed,Rows remaining
0,teams,360,0,360
1,conferences,32,0,32
2,team_conferences,4298,0,4298
3,seasons,12,0,12
4,regular_season_results,65670,326,65344
5,box_scores,131210,522,130688
6,conference_tourney_games,3785,0,3785
7,tournament_seeds,680,0,680
8,tournament_results,670,0,670
9,team_strength_preseason,4298,0,4298


### 2. Check that the files match

Next, we check whether the IDs and game records are consistent across the files. Each game should have a unique key, its teams and season should exist in the other files, and each team should have a conference for the season it played. We also check that a team plays at most once per day and that game days and locations have valid values. If a check fails, the notebook stops so we can investigate the record.


In [4]:
game_key = ["season", "day_num", "w_team_id", "l_team_id"]
keys = {
    "teams": ["team_id"], "conferences": ["conference_id"], "team_conferences": ["season", "team_id"],
    "seasons": ["season"], "regular_season_results": game_key,
    "box_scores": ["season", "day_num", "team_id"], "conference_tourney_games": game_key,
    "tournament_seeds": ["season", "team_id"], "tournament_results": game_key,
    "team_strength_preseason": ["season", "team_id"]
}
for name, key in keys.items():
    df = clean[name]
    check(f"{name}: key values present", not df[key].isna().any().any())
    check(f"{name}: key unique after exact deduplication", not df.duplicated(key).any())
    record(name, "validate keys", len(df), len(df), 0, "No missing keys or conflicting duplicate keys allowed.")

team_ids = set(clean["teams"]["team_id"])
season_ids = set(clean["seasons"]["season"])
conference_ids = set(clean["conferences"]["conference_id"])
for name, df in clean.items():
    for col in ["team_id", "opp_team_id", "w_team_id", "l_team_id"]:
        if col in df:
            check(f"{name}.{col}: valid team references", df[col].isin(team_ids).all())
    if "season" in df:
        check(f"{name}: valid season references", df["season"].isin(season_ids).all())
    if "conference_id" in df:
        check(f"{name}: valid conference references", df["conference_id"].isin(conference_ids).all())

membership = pd.MultiIndex.from_frame(clean["team_conferences"][["season", "team_id"]])
ratings = pd.MultiIndex.from_frame(clean["team_strength_preseason"][["season", "team_id"]])
check("Ratings and conference memberships cover the same team-seasons", set(membership) == set(ratings))
for name in ["regular_season_results", "tournament_results"]:
    df = clean[name]
    check(f"{name}: no self matches", (df["w_team_id"] != df["l_team_id"]).all())
    check(f"{name}: source location labels are H/A/N", df["w_loc"].isin(["H", "A", "N"]).all())
    for team_col in ["w_team_id", "l_team_id"]:
        lookup = df[["season", team_col]].rename(columns={team_col: "team_id"})
        check(f"{name}: {team_col} has season membership", pd.MultiIndex.from_frame(lookup).isin(membership).all())
    appearances = pd.concat([
        df[["season", "day_num", "w_team_id"]].rename(columns={"w_team_id": "team_id"}),
        df[["season", "day_num", "l_team_id"]].rename(columns={"l_team_id": "team_id"})
    ])
    check(f"{name}: team plays at most once per day", not appearances.duplicated().any())

lifetimes = clean["team_conferences"].merge(clean["teams"][["team_id", "first_season", "last_season"]], on="team_id", validate="many_to_one")
check("No membership before entry or after departure", lifetimes["season"].between(lifetimes["first_season"], lifetimes["last_season"]).all())
check("Regular results use days 0 to 132", clean["regular_season_results"]["day_num"].between(0, 132).all())
check("National tournament locations are neutral", clean["tournament_results"]["w_loc"].eq("N").all())
check("National tournament winner scores exceed loser scores", clean["tournament_results"]["w_score"].gt(clean["tournament_results"]["l_score"]).all())
selected = clean["teams"].loc[clean["teams"]["brief_alias"].isin(["Team A", "Team C"])]
check("Exactly one Team A and Team C", len(selected) == 2 and set(selected["brief_alias"]) == {"Team A", "Team C"})
print("Identifier, key, membership, and calendar checks passed.")


teams | validate keys: 360 -> 360 rows; affected=0. No missing keys or conflicting duplicate keys allowed.
conferences | validate keys: 32 -> 32 rows; affected=0. No missing keys or conflicting duplicate keys allowed.
team_conferences | validate keys: 4,298 -> 4,298 rows; affected=0. No missing keys or conflicting duplicate keys allowed.
seasons | validate keys: 12 -> 12 rows; affected=0. No missing keys or conflicting duplicate keys allowed.
regular_season_results | validate keys: 65,344 -> 65,344 rows; affected=0. No missing keys or conflicting duplicate keys allowed.
box_scores | validate keys: 130,688 -> 130,688 rows; affected=0. No missing keys or conflicting duplicate keys allowed.
conference_tourney_games | validate keys: 3,785 -> 3,785 rows; affected=0. No missing keys or conflicting duplicate keys allowed.
tournament_seeds | validate keys: 680 -> 680 rows; affected=0. No missing keys or conflicting duplicate keys allowed.
tournament_results | validate keys: 670 -> 670 rows; af

### 3. Check game scores against box scores

We calculate each team's points from its box score and compare them with the final score listed for that game. This helps us catch games where the winner's and loser's scores were accidentally switched. We correct a switch only when both teams' box scores confirm it. If a game still does not match, the notebook stops so we can look into it.


In [5]:
r = clean["regular_season_results"]
b = clean["box_scores"]
check("Two box-score rows per unique game", len(b) == 2 * len(r))
points = b[["season", "day_num", "team_id", "opp_team_id"]].copy()
points["points"] = 2*b["fgm"] + b["fgm3"] + b["ftm"]
w = points.rename(columns={"team_id": "w_team_id", "opp_team_id": "l_team_id", "points": "w_box_points"})
l = points.rename(columns={"team_id": "l_team_id", "opp_team_id": "w_team_id", "points": "l_box_points"})
joined = r.merge(w, on=game_key, how="left", validate="one_to_one").merge(l, on=game_key, how="left", validate="one_to_one")
check("Both box scores match every results game", not joined[["w_box_points", "l_box_points"]].isna().any().any())
match = joined["w_score"].eq(joined["w_box_points"]) & joined["l_score"].eq(joined["l_box_points"])
swapped = (joined["w_score"].eq(joined["l_box_points"]) & joined["l_score"].eq(joined["w_box_points"])
           & joined["w_box_points"].gt(joined["l_box_points"]) & ~match)
check("Every score mismatch is a verified reversal", (match | swapped).all())
score_audit = joined.loc[swapped, game_key + ["w_score", "l_score", "w_box_points", "l_box_points"]].copy()
score_audit = score_audit.rename(columns={"w_score": "original_w_score", "l_score": "original_l_score", "w_box_points": "corrected_w_score", "l_box_points": "corrected_l_score"})
r.loc[swapped.to_numpy(), ["w_score", "l_score"]] = joined.loc[swapped, ["w_box_points", "l_box_points"]].to_numpy()
check("All corrected result scores agree with box scores", r["w_score"].eq(joined["w_box_points"]).all() and r["l_score"].eq(joined["l_box_points"]).all())
check("All regular winners score more than losers", r["w_score"].gt(r["l_score"]).all())
record("regular_season_results", "repair verified score reversals", len(r), len(r), swapped.sum(), "Swap score values only when both box scores verify the correction; no game is removed.")
score_audit.to_csv(report_dir / "score_corrections.csv", index=False)
display(score_audit.head())


regular_season_results | repair verified score reversals: 65,344 -> 65,344 rows; affected=331. Swap score values only when both box scores verify the correction; no game is removed.


,season,day_num,w_team_id,l_team_id,original_w_score,original_l_score,corrected_w_score,corrected_l_score
269,2015,17,1104,1133,71,75,75,71
347,2015,18,1456,1310,64,84,84,64
356,2015,19,1151,1201,67,74,74,67
941,2015,29,1347,1293,57,66,66,57
1448,2015,40,1351,1407,65,82,82,65


### 4. Check missing and impossible box-score values

We check the box scores for missing values and counts that cannot be correct, such as negative rebounds or more field goals made than attempted. If we cannot determine the correct value, we mark that statistic as missing and record the original value. We keep the game in the dataset. Values that were already missing also stay missing rather than being treated as zero.


In [6]:
stat_cols = ["fgm", "fga", "fgm3", "fga3", "ftm", "fta", "oreb", "dreb", "ast", "tov", "stl", "blk", "pf"]
stat_audit_rows = []
b = clean["box_scores"]
box_key = ["season", "day_num", "team_id", "opp_team_id"]
missing_before = b[stat_cols].isna().sum()
for col in stat_cols:
    mask = b[col].notna() & ((b[col] < 0) | (b[col] % 1 != 0))
    for idx in b.index[mask]:
        stat_audit_rows.append({**b.loc[idx, box_key].to_dict(), "column": col, "original_value": b.at[idx, col], "reason": "Negative or fractional event count"})
    if mask.any():
        b[col] = b[col].astype(float)
        b.loc[mask, col] = float("nan")
    record("box_scores", f"validate {col} counts", len(b), len(b), mask.sum(), "Replace impossible counts with NA; preserve valid counts and all rows.")

bad_fga = b["fga"].notna() & ((b["fga"] < b["fgm"]) | (b["fga"] < b["fga3"]) | ((b["fga"]-b["fga3"]) < (b["fgm"]-b["fgm3"])))
for idx in b.index[bad_fga]:
    stat_audit_rows.append({**b.loc[idx, box_key].to_dict(), "column": "fga", "original_value": b.at[idx, "fga"], "reason": "Attempts conflict with made shots or three-point attempts; true value unavailable"})
b["fga_consistency_issue"] = bad_fga
b.loc[bad_fga, "fga"] = float("nan")
record("box_scores", "mask inconsistent field-goal attempts", len(b), len(b), bad_fga.sum(), "Keep original values in audit; leave FGA unknown instead of estimating or using zero.")
check("Three-point makes do not exceed attempts", not b["fgm3"].gt(b["fga3"]).any())
check("Three-point makes do not exceed all field-goal makes", not b["fgm3"].gt(b["fgm"]).any())
check("Free-throw makes do not exceed attempts", not b["ftm"].gt(b["fta"]).any())
stat_audit = pd.DataFrame(stat_audit_rows, columns=box_key + ["column", "original_value", "reason"])
stat_audit.to_csv(report_dir / "invalid_statistics.csv", index=False)
missing_report = pd.DataFrame({"missing_before": missing_before, "missing_after": b[stat_cols].isna().sum()})
missing_report.index.name = "column"
missing_report.reset_index().to_csv(report_dir / "box_score_missing_values.csv", index=False)
record("box_scores", "retain original missing values", len(b), len(b), 0, "No missing statistic is filled with zero or imputed.")
record("teams", "retain blank aliases", len(clean["teams"]), len(clean["teams"]), 0, "Blank brief_alias is intentional for teams other than A and C.")
display(missing_report)
display(stat_audit)


box_scores | validate fgm counts: 130,688 -> 130,688 rows; affected=0. Replace impossible counts with NA; preserve valid counts and all rows.
box_scores | validate fga counts: 130,688 -> 130,688 rows; affected=0. Replace impossible counts with NA; preserve valid counts and all rows.
box_scores | validate fgm3 counts: 130,688 -> 130,688 rows; affected=0. Replace impossible counts with NA; preserve valid counts and all rows.
box_scores | validate fga3 counts: 130,688 -> 130,688 rows; affected=0. Replace impossible counts with NA; preserve valid counts and all rows.
box_scores | validate ftm counts: 130,688 -> 130,688 rows; affected=0. Replace impossible counts with NA; preserve valid counts and all rows.
box_scores | validate fta counts: 130,688 -> 130,688 rows; affected=0. Replace impossible counts with NA; preserve valid counts and all rows.
box_scores | validate oreb counts: 130,688 -> 130,688 rows; affected=0. Replace impossible counts with NA; preserve valid counts and all rows.
box

,missing_before,missing_after
column,,
fgm,0,0
fga,2645,2651
fgm3,0,0
fga3,0,0
ftm,0,0
fta,0,0
oreb,0,0
dreb,0,6
ast,0,0


,season,day_num,team_id,opp_team_id,column,original_value,reason
0,2015,30,1339,1169,dreb,-20.0,Negative or fractional event count
1,2018,50,1231,1336,dreb,-21.0,Negative or fractional event count
2,2020,122,1290,1324,dreb,-15.0,Negative or fractional event count
3,2022,115,1241,1307,dreb,-18.0,Negative or fractional event count
4,2024,14,1439,1142,dreb,-25.0,Negative or fractional event count
5,2024,103,1156,1101,dreb,-22.0,Negative or fractional event count
6,2015,118,1291,1305,fga,15.0,Attempts conflict with made shots or three-poi...
7,2016,17,1134,1233,fga,19.0,Attempts conflict with made shots or three-poi...
8,2019,122,1432,1271,fga,25.0,Attempts conflict with made shots or three-poi...
9,2023,27,1207,1300,fga,22.0,Attempts conflict with made shots or three-poi...


### 5. Handle unreliable game locations in 2019

The data dictionary says neutral-site games were incorrectly labeled as home or away games in 2019. Since we cannot tell which labels are wrong, we keep the games but mark all 2019 locations as unknown. We save the original labels in a separate column so the change can be checked later. Other seasons keep their recorded locations.

The 2021 season had few or no spectators at many games. We keep its location labels, but will consider this when exploring home advantage.


In [7]:
r = clean["regular_season_results"]
r["w_loc_original"] = r["w_loc"]
r["location_reliable"] = r["season"].ne(2019)
bad_location = ~r["location_reliable"]
r.loc[bad_location, "w_loc"] = pd.NA
record("regular_season_results", "mask unreliable 2019 locations", len(r), len(r), bad_location.sum(), "Preserve original label; unknown venue for 2019; retain all games.")
check("Only the known 2019 season has missing result locations", r["w_loc"].isna().equals(r["season"].eq(2019)))
print("2019 games retained with unknown venue:", int(bad_location.sum()))


regular_season_results | mask unreliable 2019 locations: 65,344 -> 65,344 rows; affected=5,586. Preserve original label; unknown venue for 2019; retain all games.
2019 games retained with unknown venue: 5586


### 6. Separate regular-season and conference tournament games

The main results file already includes conference tournament games, so we match it with the conference tournament file to identify those games. We mark them in the cleaned results and also create a separate file containing regular-season games only. This lets us calculate Team A's and Team C's win rates using the same games counted in the project question.


In [8]:
r = clean["regular_season_results"]
t = clean["conference_tourney_games"]
check("Every conference tournament game appears in main results", pd.MultiIndex.from_frame(t[game_key]).isin(pd.MultiIndex.from_frame(r[game_key])).all())
r = r.merge(t[game_key].assign(is_conference_tourney=True), on=game_key, how="left", validate="one_to_one")
r["is_conference_tourney"] = r["is_conference_tourney"].eq(True)
clean["regular_season_results"] = r
record("regular_season_results", "flag conference tournaments", len(r), len(r), r["is_conference_tourney"].sum(), "Add flag using verified game keys; do not append or duplicate games.")
regular_only = r.loc[~r["is_conference_tourney"]].copy()
record("regular_season_only", "derive regular-season-only export", len(r), len(regular_only), len(r)-len(regular_only), "Analysis subset excluding conference tournaments; main cleaned results retain these games.")
print("Games flagged as conference tournaments:", int(r["is_conference_tourney"].sum()))


regular_season_results | flag conference tournaments: 65,344 -> 65,344 rows; affected=3,785. Add flag using verified game keys; do not append or duplicate games.
regular_season_only | derive regular-season-only export: 65,344 -> 61,559 rows; affected=3,785. Analysis subset excluding conference tournaments; main cleaned results retain these games.
Games flagged as conference tournaments: 3785


### 7. Check how cleaning affects the comparison

We compare Team A's and Team C's 2026 win rates at three points: in the raw data, after removing duplicate games, and after excluding conference tournament games. This shows which step changes the headline comparison. Correcting switched scores does not change who won, but it can change the point difference.

We also check how marking 2019 locations as unknown affects the location data available for analysis. We cannot recover which 2019 games were actually at neutral sites.


In [9]:
def team_records(frame, stage):
    df = frame.loc[frame["season"].eq(2026)]
    rows = []
    for team in selected.itertuples(index=False):
        wins = int(df["w_team_id"].eq(team.team_id).sum())
        losses = int(df["l_team_id"].eq(team.team_id).sum())
        games = wins + losses
        rows.append({"stage": stage, "team": team.brief_alias, "wins": wins, "losses": losses,
                     "games": games, "win_rate_pct": 100*wins/games if games else float("nan")})
    return rows
impact = pd.DataFrame(
    team_records(raw["regular_season_results"], "Raw pre-selection results") +
    team_records(clean["regular_season_results"], "Clean pre-selection results") +
    team_records(regular_only, "Clean regular-season only")
)
impact.to_csv(report_dir / "headline_before_after.csv", index=False)
display(impact.round({"win_rate_pct": 2}))

if len(score_audit):
    print("Mean margin in the corrected games, before:", round((score_audit.original_w_score-score_audit.original_l_score).mean(), 2))
    print("Mean margin in the corrected games, after:", round((score_audit.corrected_w_score-score_audit.corrected_l_score).mean(), 2))
    involved = score_audit["season"].eq(2026) & (score_audit["w_team_id"].isin(selected.team_id) | score_audit["l_team_id"].isin(selected.team_id))
    print("Score corrections involving A or C in 2026:", int(involved.sum()))
venue = []
for column, label in [("w_loc_original", "Original venue labels including 2019"), ("w_loc", "Reliable venue labels excluding 2019")]:
    loc = clean["regular_season_results"][column]
    n = int(loc.isin(["H", "A"]).sum())
    home_wins = int(loc.eq("H").sum())
    venue.append({"sample": label, "home_or_away_games": n, "home_wins": home_wins, "home_win_rate_pct": 100*home_wins/n})
venue_impact = pd.DataFrame(venue)
venue_impact.to_csv(report_dir / "location_sensitivity.csv", index=False)
display(venue_impact.round({"home_win_rate_pct": 2}))


,stage,team,wins,losses,games,win_rate_pct
0,Raw pre-selection results,Team A,22,10,32,68.75
1,Raw pre-selection results,Team C,27,4,31,87.10
2,Clean pre-selection results,Team A,22,10,32,68.75
3,Clean pre-selection results,Team C,27,4,31,87.10
4,Clean regular-season only,Team A,21,9,30,70.00
5,Clean regular-season only,Team C,24,4,28,85.71


Mean margin in the corrected games, before: -11.97
Mean margin in the corrected games, after: 11.97
Score corrections involving A or C in 2026: 0


,sample,home_or_away_games,home_wins,home_win_rate_pct
0,Original venue labels including 2019,58062,36594,63.03
1,Reliable venue labels excluding 2019,52476,33125,63.12


### 8. Save the cleaned data and check the final files

We save the cleaned datasets and a separate file containing regular-season games only. We also save reports showing the row counts, cleaning steps, and values we changed. Finally, we read the saved files back to check that their row counts and missing-value counts match what the notebook produced.

There are no national tournament seeds or results for 2020 because the tournament was canceled. The file includes 2026 seeds, but no 2026 national tournament results because those games have not been played in this dataset.


In [ ]:
seed_counts = clean["tournament_seeds"].groupby("season").size().rename("seeded_teams")
print("Seed coverage by season:")
display(seed_counts.to_frame())
print("2026 seed rows:", int(clean["tournament_seeds"]["season"].eq(2026).sum()))
print("National tournament seasons:", sorted(clean["tournament_results"]["season"].unique().tolist()))

for name, df in clean.items():
    if name in {"regular_season_results", "box_scores"}:
        filename = name + ".csv.gz"
        df.to_csv(clean_dir / filename, index=False, compression={"method": "gzip", "mtime": 0})
    else:
        filename = name + ".csv"
        df.to_csv(clean_dir / filename, index=False)
    record(name, "export", len(df), len(df), 0, f"Write {filename}; original input remains untouched.")
regular_only.to_csv(clean_dir / "regular_season_only.csv.gz", index=False, compression={"method": "gzip", "mtime": 0})

counts = pd.DataFrame([{"file": n, "raw_rows": len(raw[n]), "clean_rows": len(clean[n]),
                       "removed_exact_duplicates": len(raw[n])-len(clean[n])} for n in names])
check("Every row removal is explained by exact deduplication", all(len(raw[n])-len(clean[n]) == int(raw[n].duplicated().sum()) for n in names))
for name in names:
    filename = name + (".csv.gz" if name in {"regular_season_results", "box_scores"} else ".csv")
    exported = pd.read_csv(clean_dir / filename)
    check(f"{name}: exported row count matches", len(exported) == len(clean[name]))
    check(f"{name}: exported missing-value counts match", exported.isna().sum().equals(clean[name].isna().sum()))
pd.DataFrame(steps).to_csv(report_dir / "cleaning_steps.csv", index=False)
pd.DataFrame(checks).to_csv(report_dir / "validation_checks.csv", index=False)
counts.to_csv(report_dir / "row_counts.csv", index=False)
pd.DataFrame(inventory).to_csv(report_dir / "input_manifest.csv", index=False)

notes = f"""# Data quality notes

- Exact duplicates removed: {len(raw['regular_season_results'])-len(clean['regular_season_results'])} results rows and {len(raw['box_scores'])-len(clean['box_scores'])} box-score rows. Other files: see row_counts.csv.
- Verified score reversals repaired: {len(score_audit)} games; both reconstructed box scores support each correction. IDs are unchanged. See score_corrections.csv.
- Existing missing FGA values: {int(missing_before['fga'])}; after masking inconsistent FGA: {int(clean['box_scores']['fga'].isna().sum())}. No imputation. See invalid_statistics.csv and box_score_missing_values.csv.
- Impossible count cells made missing: {len(stat_audit)}. This includes negative rebounds and inconsistent attempts. Values cannot be reliably reconstructed.
- All {int(bad_location.sum())} games in 2019 retain their outcomes but have unknown w_loc. Original labels are in w_loc_original; location_reliable marks reliable seasons. Venue analyses must exclude unknown locations.
- fga_consistency_issue flags attempts that conflict with other shooting counts. Do not interpret missing FGA as zero; report the usable denominator for shooting features.
- Conference tournaments remain in regular_season_results.csv.gz, flagged by is_conference_tourney. They are excluded only in regular_season_only.csv.gz. Do not append conference_tourney_games to results.
- Blank brief_alias values are intentional for teams other than A and C.
- 2020 had no national tournament. Missing seeds/results for it are structural. 2021 had fewer games and low attendance, which can affect venue comparisons.
- The supplied file has {int(clean['tournament_seeds']['season'].eq(2026).sum())} seeds for 2026. The 2026 national tournament results are not yet available.
- Preseason ratings are fixed estimates, not final season rankings. Use team ID and season for joins, including conference membership.
- Corrected score margins and the headline regular-season restriction have different effects. See headline_before_after.csv and the notebook's margin output. Venue sensitivity is a sample comparison, not recovery of true 2019 locations.

All corrections are reproducible in notebooks/cleaning.ipynb. Original files are not overwritten.
"""
(report_dir / "DATA_QUALITY_NOTES.md").write_text(notes)
print("All checks passed. Cleaned files and reports saved.")


Seed coverage by season:


,seeded_teams
season,
2015,68
2016,68
2017,68
2018,68
2019,68
2021,68
2022,68
2023,68
2024,68


2026 seed rows: 0
National tournament seasons: [2015, 2016, 2017, 2018, 2019, 2021, 2022, 2023, 2024, 2025]
teams | export: 360 -> 360 rows; affected=0. Write teams.csv; original input remains untouched.
conferences | export: 32 -> 32 rows; affected=0. Write conferences.csv; original input remains untouched.
team_conferences | export: 4,298 -> 4,298 rows; affected=0. Write team_conferences.csv; original input remains untouched.
seasons | export: 12 -> 12 rows; affected=0. Write seasons.csv; original input remains untouched.
regular_season_results | export: 65,344 -> 65,344 rows; affected=0. Write regular_season_results.csv.gz; original input remains untouched.
box_scores | export: 130,688 -> 130,688 rows; affected=0. Write box_scores.csv.gz; original input remains untouched.
conference_tourney_games | export: 3,785 -> 3,785 rows; affected=0. Write conference_tourney_games.csv; original input remains untouched.
tournament_seeds | export: 680 -> 680 rows; affected=0. Write tournament_see

### Cleaning summary

The table below summarizes the changes made to the supplied data. It separates removed rows from corrected or marked values.


In [11]:
summary = pd.DataFrame([
    {"Change": "Duplicate game-result rows removed",
     "Count": len(raw["regular_season_results"]) - len(clean["regular_season_results"])},
    {"Change": "Duplicate box-score rows removed",
     "Count": len(raw["box_scores"]) - len(clean["box_scores"])},
    {"Change": "Game scores corrected using box scores",
     "Count": len(score_audit)},
    {"Change": "Box-score values marked missing",
     "Count": len(stat_audit)},
    {"Change": "2019 game locations marked unknown",
     "Count": int(bad_location.sum())},
    {"Change": "Conference tournament games excluded from regular-season-only file",
     "Count": len(clean["conference_tourney_games"])}
])

display(summary)
print(f"Games in regular-season-only file: {len(regular_only):,}")


,Change,Count
0,Duplicate game-result rows removed,326
1,Duplicate box-score rows removed,522
2,Game scores corrected using box scores,331
3,Box-score values marked missing,12
4,2019 game locations marked unknown,5586
5,Conference tournament games excluded from regu...,3785


Games in regular-season-only file: 61,559
